# Options from Zero · Episode 3: Why the price is not the expected value

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. A toy world · 4. Two opinions · 5. Free money · 6. The only price · 7. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 3,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

## 2. Last episode's question

What is \$103.67, paid in a year, worth today? Discount it at 3.60%.

In [ ]:
future = S * math.exp(r * T)
out["answer"] = {"future": future, "today": future * math.exp(-r * T)}
out["answer"]

## 3. A toy world

Keep only two outcomes: in a year the share is worth \$120 or \$80. The one-year call with strike \$105 then pays \$15 or nothing. (Episode 7 replaces this toy with a real tree.)

In [ ]:
UP, DOWN = 120.0, 80.0
tree = one_step(S, UP, DOWN, K, T, r)
out["toy"] = {"strike": K, "up": UP, "down": DOWN, "pay_up": tree["v_up"], "pay_down": tree["v_down"],
              "nodes": [{"t": 0, "k": 0, "price": S, "value": None},
                        {"t": 1, "k": 0, "price": DOWN, "value": tree["v_down"]},
                        {"t": 1, "k": 1, "price": UP, "value": tree["v_up"]}]}
out["toy"]

## 4. Two opinions

A natural guess: price = the chance of each outcome × what it pays, discounted. But people disagree about the chances. A pessimist who thinks the share rises with 30% probability, and an optimist who says 80%, get very different prices.

In [ ]:
disc = math.exp(-r * T)
naive = lambda p: disc * (p * tree["v_up"] + (1 - p) * tree["v_down"])
out["opinions"] = {"p_pessimist_pct": 30, "p_optimist_pct": 80, "pessimist": naive(0.3), "optimist": naive(0.8)}
out["opinions"]

## 5. Free money

Suppose someone pays the optimist's price for the call. Sell it to them, and copy the option yourself: buy 0.375 shares and borrow the rest (Episode 5 shows where 0.375 comes from). The copy pays exactly what you owe in both outcomes, so the difference is yours today, with no risk. That's an **arbitrage**: no money down, no risk, a sure profit.

In [ ]:
sell_price = out["opinions"]["optimist"]
shares, cash = tree["delta"], tree["cash"]          # cash < 0: borrowed
today_cf = sell_price - shares * S - cash            # receive the premium, buy shares, borrow
rows = [{"when": "today", "option": sell_price, "shares": -shares * S, "loan": -cash, "total": today_cf}]
for name, s_T, pay in (("share at $120", UP, tree["v_up"]), ("share at $80", DOWN, tree["v_down"])):
    rows.append({"when": name, "option": -pay, "shares": shares * s_T, "loan": cash * math.exp(r * T),
                 "total": -pay + shares * s_T + cash * math.exp(r * T)})
out["arb"] = {"rows": rows, "profit_today": today_cf, "shares": shares, "shares_cost": shares * S, "borrow": -cash,
              "owe_at_expiry": -cash * math.exp(r * T)}
assert all(abs(x["total"]) < 1e-12 for x in rows[1:])
pd.DataFrame(rows)

**Too cheap works too.** If someone sells the call at the pessimist's price, buy it and do the copy in reverse: sell 0.375 shares short and lend the proceeds you don't need. Again zero at expiry, a sure profit today.

In [ ]:
buy_price = out["opinions"]["pessimist"]
today_low = -buy_price + shares * S + cash          # buy the call, short the shares, lend
rows_low = [{"when": "today", "option": -buy_price, "shares": shares * S, "loan": cash, "total": today_low}]
for name, s_T, pay in (("share at $120", UP, tree["v_up"]), ("share at $80", DOWN, tree["v_down"])):
    rows_low.append({"when": name, "option": pay, "shares": -shares * s_T, "loan": -cash * math.exp(r * T),
                     "total": pay - shares * s_T - cash * math.exp(r * T)})
assert all(abs(x["total"]) < 1e-12 for x in rows_low[1:])
out["arb_low"] = {"rows": rows_low, "profit_today": today_low, "lend": -cash}
pd.DataFrame(rows_low)

## 6. The only price

Only one price leaves nobody a free lunch: the cost of the copy. Below it, buy the option and sell the copy; above it, the reverse. The chart shows the "expected value" price for every opinion from 0% to 100%, against the one price that rules out arbitrage. The opinions never enter it.

In [ ]:
ps = np.linspace(0, 1, 101)
out["only"] = {"price": tree["price"], "p": (ps * 100).round(1).tolist(), "naive": [naive(p) for p in ps],
               "flat": [tree["price"]] * len(ps),
               "ticks": [{"x": float(x), "label": f"{x:.0f}%"} for x in range(0, 101, 20)],
               "p_equal_pct": (tree["price"] / disc) / tree["v_up"] * 100,
               "p_99_naive": naive(0.99)}
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(ps * 100, out["only"]["naive"], label="expected payoff, discounted")
ax.axhline(tree["price"], color="k", ls="--", label="no-arbitrage price")
ax.set(xlabel="chance the share goes up (%)", ylabel="$"); ax.legend(); ax.grid(alpha=.3)
{k: v for k, v in out["only"].items() if k in ("price", "p_equal_pct")}

## 7. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")